# 🧪 EXERCISE: Group Behavioural Analysis

> **Instructions:** This notebook has intentional gaps — paths, loops, calculations and plot labels that you must complete. Look for `# TODO` and `???` markers. Run each cell in order once you have filled in the blanks.

## Learning objectives

- Set data paths for a multi-participant group study
- Implement session label normalisation (integer → S001 string format)
- Merge eye-tracking metrics into the behavioural DataFrame using pd.merge()
- Assign participants to Practice and Control groups
- Run a Mann-Whitney U test and compute Cohens d effect size
- Build individual-participant learning curves across 5 eye-tracking and behavioural metrics

---

# VISION Task — Group Behavioural Analysis (All Patients, All Sessions)

Compares accuracy, response time, and spatial performance across all participants and sessions.  
Identifies group-level patterns, hemifield asymmetries, and between-participant variability.

**SCOPE: all participants × all sessions**

---
## 0 Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from pathlib import Path
import glob
import re

sns.set_theme(style='whitegrid', palette='tab10', font_scale=1.15)
plt.rcParams.update({'figure.dpi': 120, 'figure.facecolor': 'white'})

# ── TODO 1: Set your data paths ───────────────────────────────────────────────
DATA_DIR         = Path('???')   # folder containing PsychoPy CSVs
GAZE_METRICS_DIR = Path('???')   # folder containing gaze metrics CSVs

MIN_ACCURACY      = 0.75
POSITION_STEP_DEG = 4.5
_REQUIRED_COLS    = {'trial_num', 'axis_deg', 'position_num', 'side', 'is_correct'}

SESSION_LABELS = {
    'S001': 'Friday',
    'S002': 'Saturday',
    'S003': 'Saturday (S3)',
}


---
## 1 Load All Participants & Sessions

In [ ]:
def load_csv(path):
    df = pd.read_csv(path, encoding='utf-8-sig').dropna(axis=1, how='all')
    missing = _REQUIRED_COLS - set(df.columns)
    if missing:
        print(f'  [skip] {Path(path).name} — missing: {missing}'); return None
    df = df.dropna(subset=['trial_num', 'is_correct']).reset_index(drop=True)
    df['trial_num']    = df['trial_num'].astype(int)
    df['is_correct']   = df['is_correct'].astype(int)
    df['position_num'] = df['position_num'].astype(int)
    df['axis_deg']     = df['axis_deg'].astype(int)
    if 'response_time_s' in df.columns and 'rt' not in df.columns:
        df['rt'] = pd.to_numeric(df['response_time_s'], errors='coerce')
    elif 'rt' not in df.columns:
        df['rt'] = np.nan
    df['axis_label']       = df['axis_deg'].map({0:'Horizontal',45:'Diagonal BL→TR',135:'Diagonal TL→BR'})
    df['eccentricity_deg'] = POSITION_STEP_DEG * df['position_num']
    df['hemifield']        = df['side'].map({'R':'Right','L':'Left'})
    if 'participant' not in df.columns:
        df['participant'] = Path(path).stem.split('_')[0]
    df['participant'] = df['participant'].astype(str)

    # ── TODO 2: Normalize the session column to 'S001' format ────────────────
    # PsychoPy stores session as integer (e.g. 1, 2, 3).
    # Convert to zero-padded string format: 1 → 'S001', 2 → 'S002', etc.
    # Also handle existing string formats like 'S1' → 'S001'.
    # Hint: use a helper function with re.match and f-string formatting.

    if 'session' in df.columns:
        def _norm(s):
            s = str(s).strip()
            # Your code here — replace ??? with the correct expressions:
            if re.match(r'^S\d+$', s):  return f'S{???:03d}'  # 'S1' → 'S001'
            if re.match(r'^\d+$', s):   return f'S{???:03d}'  # '1'  → 'S001'
            return s
        df['session'] = df['session'].apply(_norm)
    else:
        df['session'] = 'S001'
    return df

all_paths = sorted([p for p in glob.glob(str(DATA_DIR / '*VISION_task*.csv'))
                    if 'gaze_metrics' not in Path(p).name])
all_dfs = []
for i, path in enumerate(all_paths):
    df = load_csv(path)
    if df is not None:
        all_dfs.append(df)
        print(f'  Loaded {Path(path).name}: participant={df["participant"].iloc[0]}, '
              f'session={df["session"].iloc[0]}, n={len(df)}')

if not all_dfs:
    print(f'No valid VISION_task CSVs found in {DATA_DIR}')
else:
    all_df = pd.concat(all_dfs, ignore_index=True)
    print(f'\nTotal: {len(all_df)} trials, {all_df["participant"].nunique()} participants')
    print(f'Sessions: {sorted(all_df["session"].unique())}')


In [ ]:
# ── Merge gaze metrics into all_df ────────────────────────────────────────────
GAZE_METRICS_CSV = GAZE_METRICS_DIR / 'VISION_group_gaze_metrics_all.csv'

GAZE_COLS = [
    'sacc_count', 'sacc_amp_mean_deg', 'sacc_peak_vel_mean',
    'sacc_dur_mean_ms', 'sacc_latency_ms', 'first_sacc_toward_target',
    'fix_count', 'fix_dur_mean_ms', 'fix_dur_total_ms',
    'blink_count', 'blink_dur_mean_ms', 'gaze_error_deg',
]

if GAZE_METRICS_CSV.exists():
    gaze_df = pd.read_csv(GAZE_METRICS_CSV)
    gaze_df['participant'] = gaze_df['participant'].astype(str)
    gaze_df['trial_num']   = gaze_df['trial_num'].astype(int)

    def _norm_sess(s):
        s = str(s).strip()
        if re.match(r'^S\d+$', s): return f'S{int(s[1:]):03d}'
        if re.match(r'^\d+$', s):  return f'S{int(s):03d}'
        return s
    gaze_df['session'] = gaze_df['session'].apply(_norm_sess)

    keep_cols = ['participant', 'session', 'trial_num'] +                 [c for c in GAZE_COLS if c in gaze_df.columns]
    gaze_slim = gaze_df[keep_cols].drop_duplicates(subset=['participant','session','trial_num'])

    # Drop existing gaze cols to avoid _x/_y suffixes on re-run
    existing = [c for c in GAZE_COLS if c in all_df.columns]
    if existing:
        all_df = all_df.drop(columns=existing)

    # ── TODO 3: Merge gaze_slim into all_df ──────────────────────────────────
    # Use pd.DataFrame.merge() with:
    #   - join keys: ['participant', 'session', 'trial_num']
    #   - how='left'  (keep all behavioral rows, NaN where no gaze data)

    all_df = ???   # replace with your merge call

    merged_n = all_df['sacc_latency_ms'].notna().sum()
    print(f'Gaze metrics merged: {merged_n}/{len(all_df)} trials have saccade data')
else:
    print(f'⚠  {GAZE_METRICS_CSV} not found.')
    print('   Run VISION_group_gaze_metrics.ipynb first to generate it.')


---
## 2 Group Summary Table

In [ ]:
def agg_group(g):
    rt = g[g['is_correct']==1]['rt'].dropna()
    n = len(g); nc = g['is_correct'].sum()
    return pd.Series({
        'n_trials'     : n,
        'n_correct'    : nc,
        'accuracy'     : nc/n if n else np.nan,
        'pass_75pct'   : nc/n >= MIN_ACCURACY if n else False,
        'rt_mean'      : rt.mean(),
        'rt_sd'        : rt.std(),
        'rt_median'    : rt.median(),
        'acc_right'    : g[g['hemifield']=='Right']['is_correct'].mean(),
        'acc_left'     : g[g['hemifield']=='Left']['is_correct'].mean(),
        'acc_horiz'    : g[g['axis_label']=='Horizontal']['is_correct'].mean(),
        'acc_bltr'     : g[g['axis_label']=='Diagonal BL→TR']['is_correct'].mean(),
        'acc_tlbr'     : g[g['axis_label']=='Diagonal TL→BR']['is_correct'].mean(),
    })

per_part = all_df.groupby('participant').apply(agg_group).reset_index()
print('Group Summary (per participant):')
print(per_part.set_index('participant').round(3).to_string())
print(f'\nGroup accuracy: {per_part["accuracy"].mean():.1%} ± {per_part["accuracy"].std():.1%}')
print(f'Group RT:       {per_part["rt_mean"].mean():.3f} ± {per_part["rt_mean"].std():.3f} s')
print(f'Pass rate:      {per_part["pass_75pct"].sum()}/{len(per_part)} participants')

---
## 3 Accuracy Distribution

In [ ]:
pid_list = sorted(all_df['participant'].unique())
colors = sns.color_palette('tab10', len(pid_list))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot per participant
ax = axes[0]
accs = per_part.set_index('participant')['accuracy'].reindex(pid_list)*100
bars = ax.bar(range(len(pid_list)), accs.values,
              color=colors, edgecolor='k', lw=0.8)
ax.axhline(MIN_ACCURACY*100, color='red', ls='--', lw=1.5, label='75% criterion')
ax.set_xticks(range(len(pid_list))); ax.set_xticklabels(pid_list, rotation=45)
ax.set_ylim(0, 110); ax.set_ylabel('Accuracy (%)'); ax.set_title('Accuracy by participant')
ax.axhline(per_part['accuracy'].mean()*100, color='navy', ls=':', lw=1.5,
           label=f'Group mean ({per_part["accuracy"].mean():.0%})')
ax.legend()
for i, (bar, val) in enumerate(zip(bars, accs.values)):
    ax.text(bar.get_x()+bar.get_width()/2, val+1.5, f'{val:.0f}%', ha='center', fontsize=9)

# RT distribution per participant (box)
ax = axes[1]
data_rt = [all_df[(all_df['participant']==pid)&(all_df['is_correct']==1)]['rt'].dropna().values
           for pid in pid_list]
bp = ax.boxplot(data_rt, labels=pid_list, patch_artist=True, medianprops={'color':'black','lw':2})
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color); patch.set_alpha(0.7)
ax.set_ylabel('Response time (s)'); ax.set_title('RT distribution by participant')
plt.setp(ax.get_xticklabels(), rotation=45)
ax.axhline(all_df[all_df['is_correct']==1]['rt'].median(), color='navy', ls=':', lw=1.5,
           label='Group median')
ax.legend()

fig.suptitle('Group — Accuracy & RT Overview', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 4 Learning Curves — All Participants

In [ ]:
WIN = 8
fig, ax = plt.subplots(figsize=(14, 6))

for pi, pid in enumerate(pid_list):
    sub = all_df[all_df['participant']==pid].sort_values('trial_num').copy()
    sub['acc_roll'] = sub['is_correct'].rolling(WIN, min_periods=3, center=True).mean()*100
    ax.plot(sub['trial_num'], sub['acc_roll'], lw=1.8, color=colors[pi], label=pid, alpha=0.85)

ax.axhline(MIN_ACCURACY*100, color='red', ls='--', lw=1.5, label='75% criterion')
ax.set_xlabel('Trial number'); ax.set_ylabel('Rolling accuracy (%)')
ax.set_title(f'Learning curves — all participants (rolling window = {WIN} trials)')
ax.legend(title='Participant', fontsize=9); ax.set_ylim(-5, 110)
plt.tight_layout(); plt.show()

---
## 5 Shape Identification — Group

In [ ]:
if 'shape' in all_df.columns:
    shape_grp = all_df.groupby(['participant','shape'])['is_correct'].mean().unstack()*100
    rt_shape  = all_df[all_df['is_correct']==1].groupby(['participant','shape'])['rt'].mean().unstack()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    x = np.arange(len(pid_list)); w = 0.35
    for ax, (data, ylabel, title) in zip(axes, [
        (shape_grp, 'Accuracy (%)', 'Shape accuracy by participant'),
        (rt_shape,  'RT (s)',       'Shape RT by participant'),
    ]):
        for offset, shape, color in zip([-w/2, w/2], ['circle','triangle'], ['#2196F3','#FF9800']):
            vals = data.reindex(pid_list).get(shape, pd.Series([np.nan]*len(pid_list)))
            ax.bar(x+offset, vals.values, width=w, label=shape.capitalize(),
                   color=color, alpha=0.8, edgecolor='k', lw=0.7)
        if ylabel == 'Accuracy (%)':
            ax.axhline(MIN_ACCURACY*100, color='red', ls='--', lw=1.2)
            ax.set_ylim(0, 110)
        ax.set_xticks(x); ax.set_xticklabels(pid_list, rotation=45)
        ax.set_ylabel(ylabel); ax.set_title(title); ax.legend()

    fig.suptitle('Group — Shape Identification', fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()

---
## 6 Axis & Hemifield — Group

In [ ]:
axis_order  = ['Horizontal', 'Diagonal BL→TR', 'Diagonal TL→BR']
hemi_order  = ['Right', 'Left']

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Axis accuracy
ax = axes[0,0]
axis_acc_grp = all_df.groupby(['participant','axis_label'])['is_correct'].mean().reset_index()
axis_acc_grp_mean = axis_acc_grp.groupby('axis_label')['is_correct'].agg(['mean','std']).reindex(axis_order)
x = np.arange(len(axis_order))
ax.bar(x, axis_acc_grp_mean['mean']*100, yerr=axis_acc_grp_mean['std']*100,
       capsize=5, color=sns.color_palette('Set2',3), edgecolor='k', lw=0.8, alpha=0.85)
for pi, pid in enumerate(pid_list):
    vals = axis_acc_grp[axis_acc_grp['participant']==pid].set_index('axis_label')['is_correct'].reindex(axis_order)*100
    ax.scatter(x, vals.values, color=colors[pi], zorder=5, s=60, label=pid)
ax.axhline(75, color='red', ls='--', lw=1); ax.set_xticks(x)
ax.set_xticklabels(axis_order, rotation=20); ax.set_ylabel('Accuracy (%)')
ax.set_title('Axis accuracy (group mean ± SD)'); ax.legend(fontsize=8, title='Participant')

# Axis RT
ax = axes[0,1]
rt_grp = all_df[all_df['is_correct']==1].groupby(['participant','axis_label'])['rt'].mean().reset_index()
rt_mean = rt_grp.groupby('axis_label')['rt'].agg(['mean','std']).reindex(axis_order)
ax.bar(x, rt_mean['mean'], yerr=rt_mean['std'], capsize=5,
       color=sns.color_palette('Set2',3), edgecolor='k', lw=0.8, alpha=0.85)
for pi, pid in enumerate(pid_list):
    vals = rt_grp[rt_grp['participant']==pid].set_index('axis_label')['rt'].reindex(axis_order)
    ax.scatter(x, vals.values, color=colors[pi], zorder=5, s=60, label=pid)
ax.set_xticks(x); ax.set_xticklabels(axis_order, rotation=20)
ax.set_ylabel('RT (s)'); ax.set_title('Axis RT (group mean ± SD)'); ax.legend(fontsize=8, title='Participant')

# Hemifield accuracy
ax = axes[1,0]
hemi_acc_grp = all_df.groupby(['participant','hemifield'])['is_correct'].mean().reset_index()
hemi_mean = hemi_acc_grp.groupby('hemifield')['is_correct'].agg(['mean','std']).reindex(hemi_order)
x2 = np.arange(2)
ax.bar(x2, hemi_mean['mean']*100, yerr=hemi_mean['std']*100, capsize=5,
       color=['#2196F3','#FF9800'], edgecolor='k', lw=0.8, alpha=0.85)
for pi, pid in enumerate(pid_list):
    vals = hemi_acc_grp[hemi_acc_grp['participant']==pid].set_index('hemifield')['is_correct'].reindex(hemi_order)*100
    ax.scatter(x2, vals.values, color=colors[pi], zorder=5, s=60, label=pid)
ax.axhline(75, color='red', ls='--', lw=1); ax.set_xticks(x2)
ax.set_xticklabels(hemi_order); ax.set_ylabel('Accuracy (%)')
ax.set_title('Hemifield accuracy (group mean ± SD)'); ax.legend(fontsize=8, title='Participant')

# Hemifield RT
ax = axes[1,1]
hemi_rt_grp = all_df[all_df['is_correct']==1].groupby(['participant','hemifield'])['rt'].mean().reset_index()
hemi_rt_mean = hemi_rt_grp.groupby('hemifield')['rt'].agg(['mean','std']).reindex(hemi_order)
ax.bar(x2, hemi_rt_mean['mean'], yerr=hemi_rt_mean['std'], capsize=5,
       color=['#2196F3','#FF9800'], edgecolor='k', lw=0.8, alpha=0.85)
for pi, pid in enumerate(pid_list):
    vals = hemi_rt_grp[hemi_rt_grp['participant']==pid].set_index('hemifield')['rt'].reindex(hemi_order)
    ax.scatter(x2, vals.values, color=colors[pi], zorder=5, s=60, label=pid)
ax.set_xticks(x2); ax.set_xticklabels(hemi_order)
ax.set_ylabel('RT (s)'); ax.set_title('Hemifield RT (group mean ± SD)'); ax.legend(fontsize=8, title='Participant')

fig.suptitle('Group — Axis & Hemifield Analysis', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 7 Group Heatmap — Accuracy & RT by Position × Axis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (metric, label, cmap) in zip(axes, [
    ('is_correct', 'Group mean accuracy', 'RdYlGn'),
    ('rt',         'Group mean RT (s)',   'YlOrRd')
]):
    src = all_df if metric=='is_correct' else all_df[all_df['is_correct']==1]
    pivot = src.pivot_table(index='axis_label', columns='position_num',
                            values=metric, aggfunc='mean').reindex(
        ['Horizontal','Diagonal BL→TR','Diagonal TL→BR'])
    sns.heatmap(pivot, ax=ax, annot=True, fmt='.2f', cmap=cmap,
                linewidths=0.5, cbar_kws={'shrink':0.8})
    ax.set_xlabel('Position (eccentricity →)'); ax.set_ylabel('Axis')
    ax.set_title(f'{label} (N={len(pid_list)} participants)')
fig.suptitle('Group — Performance Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 8 Group Spatial Accuracy Map

In [ ]:
def get_xy(axis_deg, pos_num, side_char, step=POSITION_STEP_DEG):
    s = 1 if side_char=='R' else -1
    ecc = step*pos_num
    arad = np.radians(axis_deg)
    return s*ecc*np.cos(arad), s*ecc*np.sin(arad)

fig, ax = plt.subplots(figsize=(10, 10))
pos_acc = all_df.groupby(['axis_deg','position_num','side'])['is_correct'].agg(['mean','count']).reset_index()
for _, row in pos_acc.iterrows():
    x, y = get_xy(int(row['axis_deg']), int(row['position_num']), row['side'])
    acc = row['mean']
    n   = row['count']
    color = plt.cm.RdYlGn(acc)
    ax.scatter(x, y, s=200, color=color, zorder=3, edgecolor='k', lw=0.8)
    ax.text(x, y+1.2, f'{acc:.0%}\n(n={n})', ha='center', fontsize=6.5, color='#222')

from matplotlib.colorbar import ColorbarBase
from matplotlib.colors import Normalize
import matplotlib.cm as cm
sm = plt.cm.ScalarMappable(cmap='RdYlGn', norm=Normalize(vmin=0, vmax=1))
sm.set_array([])
fig.colorbar(sm, ax=ax, shrink=0.5, label='Accuracy')
ax.axhline(0, color='k', lw=0.5, ls=':'); ax.axvline(0, color='k', lw=0.5, ls=':')
ax.set_xlabel('Screen X (°)'); ax.set_ylabel('Screen Y (°)')
ax.set_title(f'Group Spatial Accuracy Map (N={len(pid_list)} participants)')
ax.set_aspect('equal'); plt.tight_layout(); plt.show()

---
## 9 Statistical Tests — Group Level

In [ ]:
print('='*60)
print(f'GROUP-LEVEL STATISTICAL TESTS (N={len(pid_list)} participants)')
print('='*60)

# 1. Per-participant binomial accuracy
print('\n1. Per-participant binomial test (accuracy > 75%):')
for pid in pid_list:
    sub = all_df[all_df['participant']==pid]
    n = len(sub); k = sub['is_correct'].sum()
    try:
        p_b = stats.binomtest(k, n, p=MIN_ACCURACY, alternative='greater').pvalue
    except AttributeError:
        p_b = stats.binom_test(k, n, p=MIN_ACCURACY, alternative='greater')
    print(f'   {pid}: {k}/{n} ({k/n:.1%}), p={p_b:.4f} {"✓ pass" if p_b < 0.05 else "✗"}')

# 2. Between-participant Kruskal-Wallis (RT)
print('\n2. Kruskal-Wallis (RT between participants):')
rt_groups = [all_df[(all_df['participant']==pid)&(all_df['is_correct']==1)]['rt'].dropna().values for pid in pid_list]
rt_groups = [g for g in rt_groups if len(g) >= 3]
if len(rt_groups) >= 2:
    stat_kw, p_kw = stats.kruskal(*rt_groups)
    print(f'   H={stat_kw:.3f}, p={p_kw:.4f} → {"Significant" if p_kw < 0.05 else "Non-significant"}')

# 3. Kruskal-Wallis by axis (group pooled)
print('\n3. Kruskal-Wallis (accuracy by axis, pooled group):')
axis_groups = [all_df[all_df['axis_label']==ax]['is_correct'].values
               for ax in ['Horizontal','Diagonal BL→TR','Diagonal TL→BR']]
stat_a, p_a = stats.kruskal(*[g for g in axis_groups if len(g)>0])
print(f'   H={stat_a:.3f}, p={p_a:.4f} → {"Significant" if p_a < 0.05 else "Non-significant"}')

# 4. Mann-Whitney: right vs left hemifield (group)
print('\n4. Hemifield asymmetry (group, Mann-Whitney):')
r_acc = all_df[all_df['hemifield']=='Right']['is_correct'].values
l_acc = all_df[all_df['hemifield']=='Left']['is_correct'].values
stat_h, p_h = stats.mannwhitneyu(r_acc, l_acc, alternative='two-sided')
print(f'   Right {r_acc.mean():.1%} vs Left {l_acc.mean():.1%}: U={stat_h:.0f}, p={p_h:.4f}')

# 5. Spearman: eccentricity vs RT
print('\n5. Spearman (eccentricity vs RT, group):')
ecc_rt = all_df[all_df['is_correct']==1][['eccentricity_deg','rt']].dropna()
rho, p = stats.spearmanr(ecc_rt['eccentricity_deg'], ecc_rt['rt'])
print(f'   ρ={rho:.3f}, p={p:.4f} → {"Significant" if p < 0.05 else "Non-significant"}')

---
## 10 Export

In [ ]:
out_path = 'VISION_group_analysis_all_trials.csv'
all_df.to_csv(out_path, index=False)
print(f'Saved: {out_path}  ({len(all_df)} rows, {len(pid_list)} participants)')

summary_path = 'VISION_group_analysis_summary.csv'
per_part.round(4).to_csv(summary_path, index=False)
print(f'Saved: {summary_path}')

---
## 11 Group Comparison: Practice vs Control

### Hypothesis
We split participants into two groups:- **Practice group**: Extended opportunity to practice the VISION task (multiple sessions)- **Control group**: Minimal or single-session exposure**Our prediction**: The Practice group should show:- **Shorter saccade latencies** (faster eye movement initiation)- **Smaller saccade amplitudes** (more accurate eye targeting)- **Faster button press times** (better motor response)Let's explore whether the data supports this practice effect.

In [ ]:
# ── TODO 4: Define your participant groups ────────────────────────────────────
# List participant IDs for the Practice group (multiple sessions / trained)
# and the Control group (single session / untrained).

PRACTICE_GROUP = []   # e.g. ['JF', 'MT', 'MM']
CONTROL_GROUP  = []   # e.g. ['1111', '2222', '233975', ...]

def assign_group(pid):
    if str(pid) in [str(p) for p in PRACTICE_GROUP]:
        return 'Practice'
    elif str(pid) in [str(p) for p in CONTROL_GROUP]:
        return 'Control'
    return 'Unassigned'

all_df['group'] = all_df['participant'].apply(assign_group)

grp_summary = all_df.groupby(['group', 'participant']).size().reset_index(name='n_trials')
print('Group assignment summary:')
print(grp_summary.to_string(index=False))


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Bar chart: Control vs Practice  ×  Friday (S001) vs Saturday (S002)
# Each metric gets one subplot: 4 bars = 2 groups × 2 session days
# ─────────────────────────────────────────────────────────────────────────────

DAYS = [('S001', 'Friday'), ('S002', 'Saturday')]
GROUPS = ['Control', 'Practice']
GROUP_COLORS = {'Control': '#FF6B6B', 'Practice': '#4ECDC4'}
BAR_WIDTH = 0.35

metrics_to_plot = []
metric_labels   = []
metric_ylabels  = []

if 'rt' in all_df.columns:
    metrics_to_plot.append('rt')
    metric_labels.append('Response Time (s)')
    metric_ylabels.append('RT (s)')

et_metrics = {
    'sacc_latency_ms':   'Saccade Latency (ms)',
    'sacc_amp_mean_deg': 'Saccade Amplitude (°)',
    'sacc_peak_vel_mean': 'Peak Saccade Velocity (°/s)',
}
for col, label in et_metrics.items():
    if col in all_df.columns:
        metrics_to_plot.append(col)
        metric_labels.append(label)
        metric_ylabels.append(label)

if not metrics_to_plot:
    print('⚠  No metrics found. Re-run the gaze merge cell first.')
else:
    correct_df = all_df[all_df['is_correct'] == 1].copy()

    n_met = len(metrics_to_plot)
    fig, axes = plt.subplots(1, n_met, figsize=(5.5 * n_met, 6))
    if n_met == 1:
        axes = [axes]

    for ax, metric, label, ylabel in zip(axes, metrics_to_plot, metric_labels, metric_ylabels):
        # x positions: two day-clusters separated by a gap
        # Day cluster 0 (Friday): x = 0 (Control), 0.4 (Practice)
        # Day cluster 1 (Saturday): x = 1.2 (Control), 1.6 (Practice)
        day_gap  = 0.8    # gap between day clusters
        bar_w    = 0.35
        cluster_x = {day_code: i * (2 * bar_w + day_gap)
                     for i, (day_code, _) in enumerate(DAYS)}

        xtick_pos    = []
        xtick_labels = []

        for g_idx, grp in enumerate(GROUPS):
            for day_code, day_name in DAYS:
                # Per-participant mean for this group × session
                subset = correct_df[
                    (correct_df['group']   == grp) &
                    (correct_df['session'] == day_code)
                ]
                pp = subset.groupby('participant')[metric].mean().dropna()

                x = cluster_x[day_code] + g_idx * bar_w

                if len(pp) == 0:
                    # No data (e.g. Control has no Saturday)
                    ax.bar(x, 0, width=bar_w * 0.9,
                           color=GROUP_COLORS[grp], alpha=0.2,
                           edgecolor='grey', lw=1, linestyle='--')
                    ax.text(x, 0.02, 'n/a', ha='center', va='bottom',
                            fontsize=8, color='grey', rotation=90)
                else:
                    m   = pp.mean()
                    sem = pp.std() / np.sqrt(len(pp))
                    ax.bar(x, m, width=bar_w * 0.9, yerr=sem, capsize=6,
                           color=GROUP_COLORS[grp], alpha=0.75,
                           edgecolor='k', lw=1.2)
                    # Individual participant dots
                    jx = np.random.default_rng(0).normal(x, 0.04, size=len(pp))
                    ax.scatter(jx, pp.values, color='k', alpha=0.5, s=50, zorder=5)

        # Day separator lines and x-tick labels
        for day_code, day_name in DAYS:
            day_center = cluster_x[day_code] + (len(GROUPS) - 1) * bar_w / 2
            xtick_pos.append(day_center)
            xtick_labels.append(day_name)

        ax.set_xticks(xtick_pos)
        ax.set_xticklabels(xtick_labels, fontsize=12, fontweight='bold')
        ax.set_ylabel(ylabel, fontsize=11)
        ax.set_title(label, fontsize=12, fontweight='bold')
        ax.grid(axis='y', alpha=0.3, linestyle='--')

        # Vertical separator between days
        if len(DAYS) > 1:
            sep_x = (cluster_x['S001'] + len(GROUPS) * bar_w + cluster_x['S002']) / 2
            ax.axvline(sep_x, color='grey', lw=1, linestyle=':', alpha=0.7)

    # Shared legend
    handles = [mpatches.Patch(color=GROUP_COLORS[g], alpha=0.75, label=g) for g in GROUPS]
    fig.legend(handles=handles, title='Group', fontsize=11, title_fontsize=11,
               loc='upper right', bbox_to_anchor=(1.0, 1.0))

    fig.suptitle(
        'Control vs Practice — Friday (before training) and Saturday (after training)\n'
        'mean ± SEM, individual participants shown',
        fontsize=13, fontweight='bold', y=1.03
    )
    plt.tight_layout()
    plt.show()

    # Print stats table
    print('\nGroup × Session summary (correct trials, mean ± SEM):')
    for metric, label in zip(metrics_to_plot, metric_labels):
        print(f'\n{label}:')
        tbl = (correct_df
               .groupby(['group', 'session'])[metric]
               .agg(['count', 'mean', 'std'])
               .round(3))
        tbl['sem'] = (tbl['std'] / np.sqrt(tbl['count'])).round(3)
        # Only Friday and Saturday
        tbl = tbl[tbl.index.get_level_values('session').isin(['S001', 'S002'])]
        print(tbl.rename(index={'S001': 'Friday', 'S002': 'Saturday'},
                         level='session').to_string())


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# TODO 5: Learning curves — 5 metrics, one line per participant
#
# Your task: complete the blanks (???) so each subplot shows how each
# participant changed from Friday (S001) to Saturday (S002) for one metric.
#
# Metrics to plot (in order):
#   1. Saccade Latency   — col 'sacc_latency_ms',  all trials
#   2. Saccade Duration  — col 'sacc_dur_mean_ms', all trials
#   3. Gaze Error        — col 'gaze_error_deg',   all trials
#   4. Behavioral RT     — col 'rt',               all trials
#   5. Motor RT          — col 'rt',               correct trials only
# ─────────────────────────────────────────────────────────────────────────────

GROUP_COLORS  = {'Control': '#FF6B6B', 'Practice': '#4ECDC4'}
SESSIONS_SHOW = [s for s in ['S001', 'S002'] if s in SESSION_LABELS]
x_pos         = {s: i for i, s in enumerate(SESSIONS_SHOW)}
x_labels      = [SESSION_LABELS[s] for s in SESSIONS_SHOW]

# Only participants assigned to a group (not 'Unassigned')
assigned_pids = sorted(
    p for p in all_df['participant'].unique()
    if all_df.loc[all_df['participant'] == p, 'group'].iloc[0] != 'Unassigned'
)

# ── TODO 5a: Fill in the missing metric definitions ───────────────────────────
# Each tuple: (column_name, plot_title, y_axis_label, correct_trials_only)
# correct_trials_only=True  → only use rows where is_correct == 1
# correct_trials_only=False → use all trials
METRIC_DEFS = [
    ('sacc_latency_ms',  'Saccade Latency',            'Latency (ms)',  False),
    ('sacc_dur_mean_ms', '???',                         'Duration (ms)', False),  # TODO: add title
    ('???',              'Gaze Error',                  '???',           False),  # TODO: add column + ylabel
    ('rt',               'Behavioral RT\n(all trials)', 'RT (s)',        False),
    ('???',              'Motor RT\n(correct trials)',   'RT (s)',        ???),   # TODO: column + True/False
]

# Filter to metrics that actually have data in all_df
available = [
    m for m in METRIC_DEFS
    if m[0] in all_df.columns and all_df[m[0]].notna().any()
]

if not available:
    print('⚠  No metrics found. Re-run the merge cells first.')
elif not assigned_pids:
    print('⚠  No participants assigned to groups yet. Fill in TODO 4 first.')
else:
    n_met = len(available)
    fig, axes = plt.subplots(1, n_met, figsize=(5 * n_met, 6), sharey=False)
    if n_met == 1:
        axes = [axes]

    for ax, (col, title, ylabel, correct_only) in zip(axes, available):

        # ── TODO 5b: Select the right rows ───────────────────────────────────
        # If correct_only is True  → keep only rows where is_correct == 1
        # If correct_only is False → use all rows
        # Hint: all_df[all_df['is_correct'] == 1]  vs  all_df
        src = ???   # replace with a conditional expression

        for pid in assigned_pids:
            pid_data = src[src['participant'] == pid]
            if pid_data.empty:
                continue
            group = all_df.loc[all_df['participant'] == pid, 'group'].iloc[0]
            color = GROUP_COLORS.get(group, 'grey')

            # ── TODO 5c: Compute per-session mean for this participant ─────────
            # Goal: for each session (S001, S002), get the mean value of `col`.
            # Use .groupby(key)[column].mean()
            sess_mean = (
                pid_data[pid_data['session'].isin(SESSIONS_SHOW)]
                .groupby('???')['???']   # TODO: groupby key, then column name
                .mean()
            )

            # Only plot sessions that have non-NaN data
            plot_sess = [
                s for s in SESSIONS_SHOW
                if s in sess_mean.index and pd.notna(sess_mean[s])
            ]
            if not plot_sess:
                continue

            xs = [x_pos[s] for s in plot_sess]
            ys = [sess_mean[s] for s in plot_sess]

            ax.plot(xs, ys, 'o-', color=color, lw=2.2, markersize=9, alpha=0.85, zorder=3)
            ax.text(xs[-1] + 0.05, ys[-1], pid,
                    fontsize=7.5, color=color, va='center', fontweight='bold')

        # ── TODO 5d: Add axis labels and title ───────────────────────────────
        # Each subplot should show the correct ylabel and title for its metric.
        # The variables `ylabel` and `title` already hold the right strings.
        ax.set_xticks(list(x_pos.values()))
        ax.set_xticklabels(x_labels, fontsize=11, fontweight='bold')
        ax.set_ylabel('???', fontsize=11, fontweight='bold')        # TODO: use ylabel
        ax.set_title('???',  fontsize=12, fontweight='bold', pad=8)  # TODO: use title
        ax.set_xlabel('Session', fontsize=10)
        ax.grid(True, alpha=0.3, linestyle='--')
        ax.set_xlim(-0.3, len(SESSIONS_SHOW) - 0.5)

    # Group legend (already complete — no TODO here)
    handles = [
        mpatches.Patch(color=c, alpha=0.85, label=g)
        for g, c in GROUP_COLORS.items()
    ]
    fig.legend(handles=handles, title='Group', fontsize=11, title_fontsize=11,
               loc='upper right', bbox_to_anchor=(1.0, 1.0))

    fig.suptitle(
        '???',   # TODO: write a meaningful figure title (describe what the graph shows)
        fontsize=13, fontweight='bold', y=1.04
    )
    plt.tight_layout()
    plt.show()

    print(f'Participants plotted: {assigned_pids}')
    print(f'Metrics shown: {[m[1].replace(chr(10), " ") for m in available]}')


In [ ]:
# Statistical comparison: Practice vs Control
print('=' * 70)
print('PRACTICE VS CONTROL GROUP COMPARISON')
print('=' * 70)

correct_df = all_df[all_df['is_correct'] == 1].copy()

metrics_to_test = [
    ('rt',                'Response Time (s)'),
    ('sacc_latency_ms',   'Saccade Latency (ms)'),
    ('sacc_amp_mean_deg', 'Saccade Amplitude (deg)'),
]

for metric, label in metrics_to_test:
    if metric not in correct_df.columns:
        continue

    ctrl  = correct_df[correct_df['group'] == 'Control'][metric].dropna()
    pract = correct_df[correct_df['group'] == 'Practice'][metric].dropna()

    if len(ctrl) < 2 or len(pract) < 2:
        continue

    # ── TODO 6: Run a Mann-Whitney U test and compute Cohen's d ──────────────
    # 1. Use stats.mannwhitneyu(ctrl, pract, alternative='two-sided')
    # 2. Compute Cohen's d: (mean_ctrl - mean_pract) / pooled_std
    #    pooled_std = sqrt((ctrl.std()**2 + pract.std()**2) / 2)
    # 3. Print results with a significance label (ns / * / ** / ***)

    # stat, p = ???
    # cohens_d = ???
    # sig = ???

    print(f'\n{label}:')
    print(f'  Control:  n={len(ctrl):3d}, mean={ctrl.mean():.3f}, SD={ctrl.std():.3f}')
    print(f'  Practice: n={len(pract):3d}, mean={pract.mean():.3f}, SD={pract.std():.3f}')
    # print(f'  Mann-Whitney U: U={stat:.1f}, p={p:.4f} {sig}')
    # print(f"  Cohen's d: {cohens_d:.3f}")

print('\n' + '=' * 70)